# PPO from scratch — **V6_0: Parallel rollout → dataset → minibatches**

First half of the GPU-shaped rewrite. The goal of V6_0/V6_1 is to make the training loop look
like real large-model practice: **inference batched wide** (many envs stepped as one forward
pass) and **updates batched deep** (SGD over shuffled minibatches from a rollout dataset).
V6_0 builds the entire *data half* of that pipeline — deliberately with **no critic, no
returns, no training**. That arrives in V6_1, on top of what you build here.

| Notebook | Half | Concepts |
|---|---|---|
| **V6_0 (this)** | **Data** | vector env API · autoreset masking · **persistent env state across collections** · rollout → `Dataset` → `DataLoader` minibatches · throughput |
| V6_1 | Training | returns/GAE at collection time · minibatched PPO updates · critic returns |

### Why 512 environments
During collection, the policy's forward pass sees one batch of `N_ENVS` observations per step —
the env count IS the inference batch size. A large model on an accelerator is matmul-bound:
feeding it 512 observations at a time is the difference between a busy GPU and an idle one.
(Our MLP on CPU won't saturate anything, but the *shape* of the pipeline — and the speedup over
one-env-at-a-time — is real and measured below.)

### Why a DataLoader
Real PPO doesn't take gradient steps on the whole 40k-sample rollout at once — it shuffles the
rollout into minibatches (`MINIBATCH_SIZE`) and walks them, re-shuffling every epoch. That's
what bounds memory and keeps the update phase a stream of well-sized matmuls. `Dataset` +
`DataLoader` is torch's standard machinery for exactly this walk, and V6_1's update loop will
consume it directly.

### ✅ Done-when
- The **autoreset contract cell** and the **masking + continuity test** (both given) pass —
  your collection skips padding steps, nothing else, and a second call genuinely resumes where
  the first left off instead of resetting.
- The **throughput benchmark** shows batched collection beating the one-env-at-a-time loop by
  a wide margin (order of magnitude, not percent).
- The **pipeline verify cell** passes: every collected sample comes out of the loader exactly
  once per walk, minibatches have the right shapes, and two walks yield different orders
  (shuffling is live).

> Kernel: `ppo`. Only `MyPolicy` survives from V5 — everything else in this notebook is new
> plumbing. The policy stays untrained; V6_0 only *moves data*.

## Imports & configuration *(given)*

In [1]:
import time

import numpy as np
import torch
import torch.nn as nn
from torch.distributions.categorical import Categorical
from torch.utils.data import DataLoader, TensorDataset
import gymnasium as gym
from gymnasium.vector import AutoresetMode

ENV_NAME       = "Acrobot-v1"
SEED           = 0
HIDDEN         = [64, 64]

# ---- NEW in V6_0 -------------------------------------------------------------
N_ENVS         = 512      # parallel envs = the inference batch size during collection
ROLLOUT_STEPS  = 80       # vector steps per collection: 512 x 80 = 40_960 transitions (minus padding)
MINIBATCH_SIZE = 4096     # SGD minibatch the DataLoader will serve (=> ~10 minibatches per walk)

print("gymnasium", gym.__version__, "| torch", torch.__version__)

gymnasium 1.3.0 | torch 2.13.0+cpu


## Policy *(given — your V5 `MyPolicy`, unchanged; nothing else survives)*

No critic in this notebook — there is nothing to fit. The untrained policy is only here so the
rollout has something to sample actions (and their log-probs) from, batched.

In [2]:
class MyPolicy(nn.Module):
    def __init__(self, input_size: int, output_size: int) -> None:
        super().__init__()
        sizes = [input_size] + HIDDEN + [output_size]
        layers = []
        for in_sz, out_sz in zip(sizes, sizes[1:]):
            layers.append(nn.Linear(in_sz, out_sz))
            layers.append(nn.Tanh())
        layers = layers[:-1]
        self.linear_layers = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> Categorical:
        return Categorical(logits=self.linear_layers(x))

    def greedy(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear_layers(x).argmax()

    def sample(self, x: torch.Tensor) -> torch.Tensor:
        return self(x).sample()
    
    def sample_with_logprob(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        distribution = self.forward(x)
        action = distribution.sample()
        logprob = distribution.log_prob(action)
        return action, logprob


def make_vec_env(n_envs: int = N_ENVS, env_name: str = ENV_NAME) -> gym.vector.VectorEnv:
    return gym.make_vec(env_name, num_envs=n_envs, vectorization_mode="sync", vector_kwargs={'autoreset_mode':AutoresetMode.SAME_STEP})


_ve = make_vec_env(3)
_o, _ = _ve.reset(seed=0)
OBS_DIM = _ve.single_observation_space.shape[0]
N_ACTS  = _ve.single_action_space.n
print("autoreset mode:", _ve.metadata["autoreset_mode"])
print(f"{ENV_NAME}: obs_dim={OBS_DIM}, n_acts={N_ACTS} | reset -> {_o.shape} {_o.dtype}")
_ve.close()

autoreset mode: AutoresetMode.SAME_STEP
Acrobot-v1: obs_dim=6, n_acts=3 | reset -> (3, 6) float32


### 🔬 The autoreset contract *(given — read this one carefully)*

When an episode ends, the vector env resets that slot automatically — but on the *next* step:
the step where `done` is True returns the episode's **real final observation and reward**, and
the step *after* it is a **padding step** — the action you submit is ignored, reward is 0, both
flags are False, and the returned obs already belongs to the *new* episode. Record a padding
step and you've put a fake transition into your dataset. This cell drives two envs to an
episode boundary and pins the semantics.

## 1. Batched collection *(you implement)*

Collect `n_steps` vector steps into flat tensors. Per step: one `no_grad` forward through the
policy gives actions and log-probs for all envs at once (`sample_with_logprob` already handles
a batch of observations — check the shapes and convince yourself why), then one `envs.step()`.
Keep, for every env that is **not** on a padding step this step: the observation you acted
from, the action, its log-prob, and the reward that came back. A boolean mask per step does
this without any per-env Python loop — that's the whole point of the exercise.

**Why the `state` parameter — measured, not hand-waved.** Across 2,048 measured episodes, a
random policy *never* reached Acrobot's goal before step ~277 of an episode (and 0 times inside
the first 80 steps): success needs hundreds of steps of accumulated energy. If every collection
reset its envs, an 80-step window would only ever contain steps 1–80 of fresh episodes — a
region where the goal is **unreachable** — no matter how many epochs V6_1 runs. Carrying the
envs across calls turns that permanent blindness into a short transient: by call *k* the envs
are ~80·k steps into their episodes, and from ~the 6th call the step stream is statistically
identical to collecting whole episodes. So the contract is: `state=None` means first call — do
the initial `envs.reset(seed=SEED)` yourself; otherwise resume from what you returned last
time. What lives inside `state` is your design (at minimum: the current observations and the
pending-padding mask) — the caller just threads it through.

Two notes:
- The tensors come back **flat** — `(M, 6)`, `(M,)`, … — with no trajectory structure. Order
  won't matter downstream (the loader shuffles anyway). V6_1 is where per-trajectory work
  (returns, GAE) happens *at collection time, before flattening* — foreshadowed, not needed yet.
- Also count and return how many padding steps you skipped, so
  `M + n_padding == n_envs * n_steps` is checkable from outside.

In [3]:
class RolloutBuffer:
    def __init__(self):
        self.obs_buffer = []
        self.actions_buffer = []
        self.rewards_buffer = []
        self.logp_buffer = []
        self.completed_buffer = []
        self.truncated_buffer = []

    def append(self, obs:torch.Tensor, actions: torch.Tensor, rewards: torch.Tensor, logps: torch.Tensor, completed: torch.Tensor, truncated: torch.Tensor):
        self.obs_buffer.append(obs)
        self.actions_buffer.append(actions)
        self.rewards_buffer.append(rewards)
        self.logp_buffer.append(logps)
        self.completed_buffer.append(completed)
        self.truncated_buffer.append(truncated)

    def get_data(self) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        obs = torch.stack(self.obs_buffer).reshape(-1, OBS_DIM)
        acc = torch.stack(self.actions_buffer).reshape(-1)
        rew = torch.stack(self.rewards_buffer).reshape(-1)
        logp = torch.stack(self.logp_buffer).reshape(-1)
        return obs, acc, rew, logp

    def __len__(self):
        buffers = [self.obs_buffer, self.actions_buffer, self.rewards_buffer,
                   self.logp_buffer, self.completed_buffer, self.truncated_buffer]
        lens = [len(p) * (p[0].shape[0] if len(p) else 1) for p in buffers]   
        assert all([x==lens[0] for x in lens])
        return lens[0] 

In [4]:
def collect_rollout(envs: gym.vector.VectorEnv, policy_net: MyPolicy, n_steps: int,
                    state: dict | None = None,
                    ) ->  tuple[dict, RolloutBuffer]:
    # TODO (V6_0): n_steps batched steps as described above. state=None -> first call:
    # reset the envs (seed=SEED) and start fresh; otherwise resume from the state you
    # returned last time. Return (obs (M,6), act (M,), logp (M,), rew (M,), n_padding, state),
    # where M = n_envs * n_steps - n_padding.
    if state is None:
        initial_obs = torch.tensor(envs.reset(seed=SEED)[0], dtype=torch.float32)
        state = {'obs': initial_obs}
    buffer = RolloutBuffer() 
    for step in range(n_steps):
        obs = state['obs']
        with torch.no_grad():
            actions, logps =policy_net.sample_with_logprob(obs)
        next_obs, rews, termin, trunc, info = envs.step(actions.cpu().numpy())
        next_obs, rews = torch.tensor(next_obs, dtype=torch.float32), torch.tensor(rews, dtype=torch.float32)
        termin, trunc = torch.tensor(termin), torch.tensor(trunc)
        buffer.append(obs, actions, rews, logps,termin, trunc)
        done = termin | trunc
        if done.any():
            final_idxs = info['_final_obs']
            assert (final_idxs == done).all()
            final_obs =  np.stack(info['final_obs'][final_idxs])
            final_vals = torch.zeros(final_obs.shape[0]) 
            # TODO
            
        state['obs'] = next_obs
      
    return state, buffer

### 🔬 Masking + continuity test *(given)*

A fresh Acrobot can't end inside 80 steps (truncation is at 500), so your headline collection
sees almost no padding. This test runs 8 envs for 600 total steps — but as **two 300-step calls
with the state threaded through**. If the state genuinely carries, every env's step-500
truncation (and its padding step) must land in the *second* window; if your second call
secretly resets, the second window looks as padding-free as the first and the test fails.

In [5]:
def _test_masking_and_continuity() -> None:
    torch.manual_seed(0)
    e = make_vec_env(8)
    p = MyPolicy(OBS_DIM, N_ACTS)
    st, buff1 = collect_rollout(e, p, 300)        # episode steps 1-300
    M1 = len(buff1)
    obs1, act1, rew1, logp1 = buff1.get_data()
    st, buff2 = collect_rollout(e, p, 300, st)    # episode steps 301-600
    obs2, act2, rew2, logp2 = buff2.get_data()
    M2 = len(buff2)
    e.close()
    assert (M1,M2) == (300 * 8,300*8), "accounting broken: samples != 2400"
    
    for obs, act, logp, rew, M in ((obs1, act1, logp1, rew1, M1), (obs2, act2, logp2, rew2, M2)):
        assert act.shape == logp.shape == rew.shape == (M, ), "steps times num envs"
        assert obs.shape == (M, OBS_DIM)
        assert (logp <= 0).all(), "log-probs must be <= 0"
    vals = set(torch.cat([rew1, rew2]).unique().tolist())
    assert vals <= {-1.0, 0.0}, f"Acrobot rewards are -1 (step) or 0 (goal); got {vals}"
    epoch1_trunc = torch.cat(buff1.truncated_buffer).sum()
    epoch1_complete = torch.cat(buff1.completed_buffer).sum()
    epoch2_trunc = torch.cat(buff2.truncated_buffer).sum()
    epoch2_complete = torch.cat(buff2.completed_buffer).sum()
    assert epoch1_trunc == 0, "there should be no truncation in epoch0"
    assert epoch1_complete + epoch2_complete <= 2, "suspciously many complete"
    assert epoch1_complete + epoch2_complete + epoch1_trunc + epoch2_trunc == 8, "all lanes should've finished once"

_test_masking_and_continuity()

## 2. Throughput — the point of all this *(given)*

Same `collect_rollout`, two shapes: one env stepped ~2000 times vs 512 envs stepped 80 times.
Identical code path, so the ratio is purely the win from batching the forward pass and the env
stepping.

In [6]:
torch.manual_seed(SEED)
_bench_policy = MyPolicy(OBS_DIM, N_ACTS)

e1 = make_vec_env(1)
t0 = time.time()
state, buffer = collect_rollout(e1, _bench_policy, 2000)
dt1 = time.time() - t0
e1.close()

sps_single = len(buffer) / dt1

eN = make_vec_env(N_ENVS)
t0 = time.time()
state2, bufferN = collect_rollout(eN, _bench_policy, ROLLOUT_STEPS)
dtN = time.time() - t0
eN.close()
sps_batched = len(bufferN) / dtN

print(f"single env : {len(buffer):6d} samples in {dt1:5.2f}s  -> {sps_single:9.0f} samples/s")
print(f"{N_ENVS:4d} envs  : {len(bufferN):6d} samples in {dtN:5.2f}s  -> {sps_batched:9.0f} samples/s")
print(f"speedup: {sps_batched / sps_single:.1f}x  (same code, batched forward + batched stepping)")

single env :   2000 samples in  1.19s  ->      1686 samples/s
 512 envs  :  40960 samples in  3.16s  ->     12950 samples/s
speedup: 7.7x  (same code, batched forward + batched stepping)


## 3. Dataset → DataLoader *(you implement)*

Wrap the four aligned tensors in a `TensorDataset` and serve them through a `DataLoader` with
`batch_size=MINIBATCH_SIZE` and shuffling **on** — a shuffled loader re-draws the order on
every walk, which is exactly what the PPO update epochs in V6_1 need (fresh minibatch
composition each pass). One conscious decision: with zero padding, 40 960 splits into exactly
ten 4096-minibatches — but any padding makes the last minibatch partial. Keep it (default) or
`drop_last` — either is defensible; know which you chose.

In [7]:
def make_loader(rollout_buffer:RolloutBuffer,
                batch_size: int = MINIBATCH_SIZE) -> DataLoader:
    
    obs, act, rew, logp1 = rollout_buffer.get_data()
    ds = TensorDataset(obs, act, rew, logp1)
    return DataLoader(ds, batch_size=batch_size, shuffle=True)

## ✅ Verify V6_0 *(given)*

The full pipeline: collect with 512 envs, load, walk. Every sample must come out exactly once
per walk, and two walks must disagree on order.

In [8]:
torch.manual_seed(SEED)
policy_net = MyPolicy(OBS_DIM, N_ACTS)

# THE V6_1 PATTERN: one persistent vector env, `state` threaded through the epoch loop --
# each epoch collects a fresh dataset while the episodes keep running underneath.
envs = make_vec_env()
state = None
for epoch in range(2):
    state, buffer = collect_rollout(envs, policy_net, ROLLOUT_STEPS, state)
    print(f"epoch {epoch}: {len(buffer)} transitions "
          f"envs now ~{(epoch + 1) * ROLLOUT_STEPS} steps into their episodes")
envs.close()

M = len(buffer)
obs, act, rew, logp1 = buffer.get_data()
loader = make_loader(buffer)
sizes, first = [], None
for mb_obs, mb_act, mb_rew, mb_logp in loader:
    if first is None:
        first = mb_obs.sum().item()
    assert mb_obs.shape[1:] == (OBS_DIM,) and mb_act.shape == mb_logp.shape == mb_rew.shape
    sizes.append(mb_obs.shape[0])
assert sum(sizes) == M, f"loader walk saw {sum(sizes)} samples, dataset has {M}"
print(f"one walk: {len(sizes)} minibatches, sizes {sizes}")

second = next(iter(loader))[0].sum().item()
assert first != second, "two walks produced the same first minibatch -- shuffling is off"
print("re-shuffle across walks: OK")

hist = torch.bincount(act, minlength=N_ACTS).float() / M
print(f"action distribution (untrained policy, ~uniform): {[f'{p:.3f}' for p in hist.tolist()]}")
print(f"rewards: {int((rew == -1).sum())} steps at -1, {int((rew == 0).sum())} goal steps at 0")
print("\nV6_0 pipeline verified: this loader is exactly what V6_1's update loop will consume.")

epoch 0: 40960 transitions envs now ~80 steps into their episodes


epoch 1: 40960 transitions envs now ~160 steps into their episodes


one walk: 10 minibatches, sizes [4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096]
re-shuffle across walks: OK
action distribution (untrained policy, ~uniform): ['0.363', '0.284', '0.353']
rewards: 40960 steps at -1, 0 goal steps at 0

V6_0 pipeline verified: this loader is exactly what V6_1's update loop will consume.


---
*V6_1 adds the training half: the critic returns, returns/GAE get computed per trajectory at
collection time (before flattening), and the PPO update becomes a minibatched walk over this
loader — the CleanRL/SB3 shape.*